# MDS — Fuerza bruta

Enumeración de los $2^n$ subconjuntos; se conserva el de mínimo tamaño que domina el grafo.
Baseline **clásico exacto**.


In [1]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


In [2]:
nodes = ["Mario", "Sarah", "Raul", "Ana", "Enrique", "Saul"]
edges = [("Mario", "Sarah"), ("Mario", "Raul"), ("Mario", "Ana"),
         ("Enrique", "Sarah"), ("Enrique", "Raul"), ("Saul", "Ana")]
idx = {v: i for i, v in enumerate(nodes)}
n = len(nodes)
adj = {i: set() for i in range(n)}
for a, b in edges:
    adj[idx[a]].add(idx[b])
    adj[idx[b]].add(idx[a])
closed = {v: sorted(adj[v] | {v}) for v in range(n)}

def bits_cost_payments(bits):
    return int(sum(bits))

def is_dominating(bits):
    return all(any(bits[u] for u in closed[v]) for v in range(n))

def decode(bits):
    return [nodes[i] for i in range(n) if bits[i]]

# Óptimo exacto (referencia compartida)
best_cost, best_bits = n + 1, None
for k in range(2 ** n):
    bits = [(k >> i) & 1 for i in range(n)]
    if is_dominating(bits):
        c = bits_cost_payments(bits)
        if c < best_cost:
            best_cost, best_bits = c, bits
OPTIMAL_COST = float(best_cost)
OPTIMAL_SOL = decode(best_bits)
print("Óptimo MDS:", OPTIMAL_SOL, "coste =", OPTIMAL_COST)

def resolve_metrics_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization"
        if root.is_dir():
            d = root / "metrics"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path.cwd().resolve() / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization" / "metrics"
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_mds_metrics(payload: dict, filename: str):
    path = resolve_metrics_dir() / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


Óptimo MDS: ['Ana', 'Enrique'] coste = 2.0


## Resolución y métricas


In [3]:
t0 = time.perf_counter()
best_c, best_b, evaluated = n + 1, None, 0
for k in range(2 ** n):
    bits = [(k >> i) & 1 for i in range(n)]
    evaluated += 1
    if is_dominating(bits):
        c = bits_cost_payments(bits)
        if c < best_c:
            best_c, best_b = c, bits
elapsed = time.perf_counter() - t0
sol = decode(best_b)
success = abs(best_c - OPTIMAL_COST) < 1e-9
print("Solución:", sol, "coste =", best_c)

payload = {
    "problem": "mds",
    "solver": "brute_force",
    "family": "classical",
    "protocol": {"n_runs": 1, "method": "enumerate_2^n"},
    "instance": {"nodes": nodes, "edges": [list(e) for e in edges]},
    "quality": {
        "optimal_cost": OPTIMAL_COST,
        "optimal_solution": OPTIMAL_SOL,
        "cost_mean": float(best_c),
        "cost_best": float(best_c),
        "gap_mean": float(best_c - OPTIMAL_COST),
    },
    "success_rate": float(success),
    "time_s": {"mean": float(elapsed), "std": 0.0, "total": float(elapsed)},
    "effort": {"subsets_evaluated": evaluated},
    "runs": [{"seed": 0, "solution": sol, "cost": float(best_c), "success": bool(success), "time_s": float(elapsed)}],
}
save_mds_metrics(payload, "bruteforce_mds.json")


Solución: ['Ana', 'Enrique'] coste = 2
Saved: /Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/bruteforce_mds.json


PosixPath('/Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/bruteforce_mds.json')

## Storytelling plots

Figuras para documento/slides → `figures/`.


In [ ]:
# --- storytelling plots (auto) ---
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

for _base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _lib = _base / "Notebooks" / "lib"
    if (_lib / "storytelling.py").is_file():
        _s = str(_lib)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
else:
    raise FileNotFoundError("Notebooks/lib/storytelling.py not found")

import importlib
import storytelling as _storytelling
importlib.reload(_storytelling)
from storytelling import (
    resolve_figures_dir,
    plot_mds_instance, plot_mds_solution,
)

figures_dir = resolve_figures_dir("Optimization", "MinimumDominatingSetOptimization")
plot_mds_instance(nodes, edges, figures_dir, stem="instance")
_dom = None
if "payload" in globals() and payload.get("runs"):
    _runs = [r for r in payload["runs"] if r.get("solution") is not None]
    if _runs:
        _dom = min(_runs, key=lambda r: float(r.get("cost", 1e18))).get("solution")
if _dom is None and "sol" in globals():
    _dom = sol
if _dom is None and "OPTIMAL_SOL" in globals():
    _dom = OPTIMAL_SOL
if _dom is None and "best_b" in globals() and best_b is not None:
    _dom = decode(best_b)
if _dom is None:
    raise RuntimeError("No MDS dominating set found for storytelling plot")
plot_mds_solution(nodes, edges, _dom, figures_dir, stem="bruteforce", title="BruteForce")

plt.show()
